MCP allows re-usability for commonly used tools across different agentic workflows.


# MCP specs

* Based on JSON-RPC 2.0

* Parties involved
    - Host: LLM application that initiates connection
    - Client: Connector with host application
    - Server: MCP server which provides the context

* Server provides any of the following context services
    - Resources - Static/dynamic content like data/content
    - Tools - Functions which execute something
    - Prompts - Templated messages and workflows for users

* Clients may offer following features to the server
    - Sampling: Server initiated agentic behavious and recursive LLM interactions
    - Roots: Server initiated inquiries into uri or filesystem boundaries to operate in
    - Elicitation: Server initiated requests for additional info from users

* Additional utilities(based off context)
    - Configuration
    - Progress tracking
    - Cancellation
    - Error reporting
    - Logging

---

# MCP lifecycle:

* Initialization: Client and server negotiate capabilities & protocol version alignment; 
  Client initiates communication with
    - Protocol version supported
    - Capabilities - sampling, root thing and elicitation
    - Implementation info
Server reverts with its capabilities & implementation info
Client sends notification to server "initialized"
Singleton connection object creation of client and given to host

* Operation: Normal protocol communication

* Shutdown: Graceful termination of connection
---

# Client capabilities

These are negotiated during the initialization phase

* roots = listChanged as `true`: Sends a notification to the server whenever the list of roots change
* sampling: Lets the MCP server ask the MCP client to run a response completion request to the LLM and return the result; user has to explicitly allow control. This is required when the MCP server tool needs LLM for intelligence, for e.g. to summarise its response, draft an email for a identified issue etc.
* elicitation: ask the user for input
---

# Server capabilities

These are negotiated during the initialization phase

* Prompts: User-controlled, exposed to the user to be able to use prompt tempalates for working with the MCP server tools etc.. Similarly like roots, it notifies the client when the list of prompts change.
* Resources: App-driven, to provide things to the host application and determine how to use them, e.g. schema for a tool; similar notification
* Tools: Host-driven, to enable the LLM choose a tool if reqd, from available tools in the MCP server

---

fastMCP

fastMCP is the pythonic framework for implementing MCP

---

`FastMCP` is the core class for implementing the server. Its constructor accepts many arguments. Tools, resources and prompts can also be supplied to it. Other notable params - auth, lifespan

Server components can be filtered basis tags for inclusion/exclusion

The MCP routes can be simply mounted as routes in a fastAPI application also.

---

# Server

Tools

* Function name -> Tool name
* Function docstring -> Tool description
* Input schema based on input params
* Output schemas as mentioned
* Handles param validation and error reporting

To override these, define them in the @mcp.tool decorator
Annotation can also be added, but that is for the user and is not used by the LLM(host)

* Use decorator @make_async_background to run a tool which is CPU heavy

Use dependency injection of `Context` to use MCP server runtime context capabilities - logging, state management, progress reporting, LLM sampling, resource access, information requests

For duplicate tools -> warn

---

Resources

Resources provide read-only access to data for the LLM or client application.

For dynamic content, define a function for it and implement it.

For static content, use appropriate subclass like `TextResource` and define the resource using `mcp.add_resource(...)`


When a client requests a resource URI:
1. FastMCP finds the corresponding resource definition.
2. If it’s dynamic (defined by a function), the function is executed.
3. The content (text, JSON, binary data) is returned to the client.
This allows LLMs to access files, database content, configuration, or dynamically generated information relevant to the conversation.

@mcp.resource("resource://{user}/greeting")
def get_greeting(user:str) -> str:
    """Provides a simple greeting message to a user."""
    return f"Hello from FastMCP Resources!, {user}"

-   resource://greeting is the unique URI used by the client to request for the resource
-   get_greeting() is loaded 'lazily' when the client requests for it
-   the resource name is get_greeting & the docstring is the description

Similarly additional params can be added in the @mcp.resource decorator. It can also use the `Context` via dependency injection.

For duplicate resources -> warn

---

Prompts

Parameterized prompt templates for the LLM to use, to generate structured output.

Prompt name -> Function name
Prompt description -> Function docstring
Prompt parameters -> Input params to the prompt function

@mcp.prompt
def generate_code_request(language: str, task_description: str) -> PromptMessage:
    """Generates a user message requesting code generation."""
    content = f"Write a {language} function that performs the following task: {task_description}"
    return PromptMessage(role="user", content=TextContent(type="text", text=content))

For duplicate prompts -> update

---


# MCP servers - advanced configurations

Composing servers

Importing vs mounting -> Importing is faster and reco for prod setups; however, mounting is dynamic and works when a modular setup is required[bottleneck is the slowest server]
In import_server(), sub-servers are imported once at startup
In mount_server(), sub-servers are added during runtime


Context for

* Logging
* LLM sampling
* Progress reporting
* Resource access
* User elicitation
* State mgmt.
* Request info
* MCP server instance access


It also supports background tasks(in-memory/redis), state mgmt.(middleware), elicitation, LLM sampling etc.

---

# Client

The client automatically determines the transport based on the URI of the MCP server - `http//`/`.py`/`.js`
The client supports callbacks for logging, LLM sampling, progress monitoring & roots monitoring to servers

There are different apis for tool/resource/prompt access like list_tools(), call_tool(), list_resources(), read_resource(), list_prompts(), get_prompt(). Each API has a return JSON and associated fields.


There are also advanced things like message handling, user elicitation, LLM sampling, roots access etc. for the client